In [1]:
!pip install -U google-genai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.0/958.0 kB 3.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 4.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [google-genai] [google-genai]


In [4]:
import os
import sys
from dotenv import load_dotenv
from google import genai


def format_token_limit(value: int | None) -> str:
    """토큰 한도를 보기 좋게 출력합니다."""
    if value is None:
        return "-"

    return f"{value:,}"


def main() -> None:
    # 프로젝트 루트의 .env 로드
    load_dotenv()

    api_key = os.getenv("API_KEY")

    if not api_key:
        raise RuntimeError(
            "GEMINI_API_KEY를 찾을 수 없습니다.\n"
            ".env 파일에 GEMINI_API_KEY=... 형식으로 설정하세요."
        )

    try:
        client = genai.Client(api_key=api_key)

        # API 키로 접근 가능한 전체 모델 조회
        models = list(client.models.list())

    except Exception as error:
        print("\n[Gemini API 연결 실패]")
        print(f"{type(error).__name__}: {error}")
        sys.exit(1)

    if not models:
        print("조회 가능한 모델이 없습니다.")
        return

    print(f"\n조회된 모델 수: {len(models)}")
    print("=" * 110)

    generation_models = []
    embedding_models = []

    for model in models:
        model_name = getattr(model, "name", "-").replace("models/", "")
        actions = getattr(model, "supported_actions", []) or []

        if "generateContent" in actions:
            generation_models.append(model_name)

        if "embedContent" in actions:
            embedding_models.append(model_name)

        print(f"모델명          : {model_name}")
        print(f"표시명          : {getattr(model, 'display_name', '-')}")
        print(f"지원 기능       : {', '.join(actions) if actions else '-'}")
        print(f"입력 토큰 한도  : {format_token_limit(getattr(model, 'input_token_limit', None))}")
        print(f"출력 토큰 한도  : {format_token_limit(getattr(model, 'output_token_limit', None))}")

        description = getattr(model, "description", None)
        if description:
            print(f"설명            : {description}")

        print("-" * 110)

    print("\n[텍스트·멀티모달 생성 가능 모델: generateContent]")
    for model_name in sorted(generation_models):
        print(f"- {model_name}")

    print("\n[임베딩 생성 가능 모델: embedContent]")
    for model_name in sorted(embedding_models):
        print(f"- {model_name}")


if __name__ == "__main__":
    main()


조회된 모델 수: 55
모델명          : gemini-2.5-flash
표시명          : Gemini 2.5 Flash
지원 기능       : generateContent, countTokens, createCachedContent, batchGenerateContent
입력 토큰 한도  : 1,048,576
출력 토큰 한도  : 65,536
설명            : Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.
--------------------------------------------------------------------------------------------------------------
모델명          : gemini-2.5-pro
표시명          : Gemini 2.5 Pro
지원 기능       : generateContent, countTokens, createCachedContent, batchGenerateContent
입력 토큰 한도  : 1,048,576
출력 토큰 한도  : 65,536
설명            : Stable release (June 17th, 2025) of Gemini 2.5 Pro
--------------------------------------------------------------------------------------------------------------
모델명          : gemini-2.0-flash
표시명          : Gemini 2.0 Flash
지원 기능       : generateContent, countTokens, createCachedContent, batchGenerateContent
입력 토큰 한도  : 1,048,576
출

In [6]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(api_key=os.environ["API_KEY"])

for model in client.models.list():
    model_name = model.name.replace("models/", "")
    actions = model.supported_actions or []

    if "generateContent" not in actions:
        continue

    try:
        response = client.models.generate_content(
            model=model_name,
            contents="Reply with exactly: OK",
        )

        print(f"✅ {model_name}")
        print(f"   응답: {response.text[:100]!r}")

    except Exception as e:
        print(f"❌ {model_name}")
        print(f"   {type(e).__name__}: {e}")

✅ gemini-2.5-flash
   응답: 'OK'
✅ gemini-2.5-pro
   응답: 'OK'
❌ gemini-2.0-flash
   ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
❌ gemini-2.0-flash-001
   ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash-001 is no longer available. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
❌ gemini-2.0-flash-lite-001
   ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash-lite-001 is no longer available. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
❌ gemini-2.0-flash-lite
   ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash-lite is no lon

✅ lyria-3-clip-preview
   응답: '[4.0:8.0] OK.\n[16.0:20.0] OK.\n[24.0:28.0] OK.'
✅ lyria-3-pro-preview
   응답: '[[A0]]\n[[B1]]\n[2.0:] OK.\n[:] (OK.)\n[:] OK.\n[:] (OK.)\n[:] OK.\n[[C2]]\n[6.0:] OK!\n[[D3]]\n[8.0:] OK.'
❌ gemini-3.1-flash-tts-preview
   ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Request contains an invalid argument.', 'status': 'INVALID_ARGUMENT'}}
❌ gemini-robotics-er-1.5-preview
   ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-robotics-er-1.5-preview is no longer available. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
✅ gemini-robotics-er-1.6-preview
   응답: 'OK'
❌ gemini-2.5-computer-use-preview-10-2025
   ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'This model requires the use of the Computer Use tool. See https://ai.google.dev/gemini-api/docs/computer-use#send-request for instructions on adding the tool. 

| 모델 ID                                | 공식 상태   | 공식 문서 기준 용도                                                                           | 프로젝트 활용 판단               |
| ------------------------------------ | ------- | ------------------------------------------------------------------------------------- | ------------------------ |
| `gemini-2.5-flash`                   |   Stable  | 가격 대비 성능 중심 모델. 저지연·대량 처리·추론·에이전트 작업에 적합. ([Google AI for Developers][1])             | **기본 LLM Judge 추천**      |
| `gemini-2.5-pro`                     |   Stable  | 코드·수학·STEM·대규모 문서/데이터 분석 같은 복잡한 추론 작업용. ([Google AI for Developers][2])               | 경계 사례 재판정, 평가 기준 검증      |
| `gemini-2.5-flash-lite`              |   Stable  | 대량 분류, 단순 정보 추출, 저비용·저지연 작업에 적합. ([Google AI for Developers][3])                      | 1차 대량 라벨링·후보 추출          |
| `gemini-3.5-flash`                   |   Stable  | 고속·저비용의 장기 에이전트 워크플로, 복잡한 코딩·반복 작업에 최적화된 최신 Flash 계열. ([Google AI for Developers][4]) | `2.5-flash`와 품질·비용 비교 후보 |
| `gemini-3.1-flash-lite`              |   Stable  | 고빈도 경량 작업, 데이터 추출, 대량 에이전트 워크플로에 최적화. ([Google AI for Developers][5])                 | 대량 JSON 추출·간단 분류         |
| `gemini-3-flash-preview`             |   Preview | 멀티모달 이해, 도구 사용, Computer Use 등을 포함한 고성능 Flash 실험 모델. ([Google AI for Developers][6])  | 실험용. 운영 기준 모델로는 보류       |
| `gemini-3.1-pro-preview`             |   Preview | 정밀한 도구 사용, 다단계 실행, 소프트웨어 엔지니어링 및 에이전트 작업에 최적화. ([Google AI for Developers][7])        | 복잡한 분석 실험·고난도 재검토        |
| `gemini-3.1-pro-preview-customtools` |   Preview | Bash·파일 탐색·검색 등 **커스텀 도구 호출 우선순위**를 높인 별도 엔드포인트. ([Google AI for Developers][7])      | 일반 LLM Judge에는 불필요       |

[1]: https://ai.google.dev/gemini-api/docs/models/gemini-2.5-flash "Gemini 2.5 Flash  |  Gemini API  |  Google AI for Developers"
[2]: https://ai.google.dev/gemini-api/docs/models/gemini-2.5-pro "Gemini 2.5 Pro  |  Gemini API  |  Google AI for Developers"
[3]: https://ai.google.dev/gemini-api/docs/models/gemini-2.5-flash-lite "Gemini 2.5 Flash-Lite  |  Gemini API  |  Google AI for Developers"
[4]: https://ai.google.dev/gemini-api/docs/models/gemini-3.5-flash "Gemini 3.5 Flash  |  Gemini API  |  Google AI for Developers"
[5]: https://ai.google.dev/gemini-api/docs/models/gemini-3.1-flash-lite "Gemini 3.1 Flash-Lite  |  Gemini API  |  Google AI for Developers"
[6]: https://ai.google.dev/gemini-api/docs/models/gemini-3-flash-preview "Gemini 3 Flash Preview  |  Gemini API  |  Google AI for Developers"
[7]: https://ai.google.dev/gemini-api/docs/models/gemini-3.1-pro-preview "Gemini 3.1 Pro Preview  |  Gemini API  |  Google AI for Developers"
